
# Validation screen: guide-level EC differentiation effect and significance

This notebook re-analyzes the **raw validation-screen guide counts** from the D3 CDH5-positive and D3 CDH5-negative fractions.

## Primary quantity

Because no FACS percentage or positive/negative fraction cell count is available, the raw sequencing data do **not** identify an absolute EC percentage. The primary per-guide result is therefore

\[
L_j = \log_2\left[
\frac{(f_{j,+}/f_{j,-})}
     {(f_{\mathrm{control},+}/f_{\mathrm{control},-})}
\right],
\]

and its more intuitive transformation

\[
\mathrm{relative\ CDH5^+\ odds\ (\%\ of\ control)}_j
=100\times 2^{L_j}.
\]

Interpretation:

- 100%: the same CDH5-positive odds as control guides;
- 50%: half the control odds;
- 200%: twice the control odds.

This must not be described as literal “percent EC.” If a defensible control EC fraction becomes available later, the notebook can convert the relative odds to a calibrated absolute estimate.

## Statistical analysis

The notebook:

1. reads the raw integer guide-count matrix;
2. maps samples to biological replicate and fraction;
3. **sums PCR/sequencing technical replicates** within each biological replicate and fraction;
4. derives sample normalization factors from safe-targeting and non-targeting controls;
5. fits an edgeR quasi-likelihood negative-binomial model

\[
\text{count} \sim \text{biological replicate} + \text{CDH5 fraction};
\]

6. tests the `POS − NEG` coefficient for each guide;
7. applies Benjamini–Hochberg correction across QC-passing targeting guides;
8. reports biological-replicate effect estimates and a direction-concordance flag.

The guide-level test is distinct from the earlier gene-level Mann–Whitney analysis across approximately six guides per target.



## Information carried forward from the existing analysis

The supplied R scripts establish the following conventions, which this notebook preserves:

- the differentiation comparison is **D3 CDH5-positive versus D3 CDH5-negative**;
- `Neg` is the first/reference condition and `Pos` is the second condition;
- negative log2 fold-change means depletion from the CDH5-positive fraction;
- MAGeCK sgRNA summaries contain guide-level `LFC` and `FDR` fields;
- the existing plots use an FDR threshold of 0.05;
- D3-total versus D0 is a separate survival/fitness comparison and is not substituted for EC differentiation efficiency.

The old R files are primarily downstream summary/plotting scripts. They do not expose a reliable validation-screen sample map, so the notebook validates inferred sample labels and writes an editable sample-sheet template whenever the names are ambiguous.



## Sherlock environment

A GPU is not needed. An interactive job with approximately 4 CPUs and 16–32 GB RAM is sufficient.

Typical setup:

```bash
srun --partition=interactive --time=04:00:00 --mem=32G --cpus-per-task=4 --pty bash
module spider R
module load R
jupyter lab --no-browser
```

The Python kernel needs `pandas`, `numpy`, and `matplotlib`. The R installation needs `edgeR`. A later cell checks this and prints an installation command when required.


In [ ]:

from __future__ import annotations

from pathlib import Path
from typing import Iterable, Optional
import json
import math
import os
import re
import shutil
import subprocess
import textwrap
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

print("Python:", os.sys.version.split()[0])
print("User:", os.environ.get("USER", "unknown"))


## 1. Configuration — edit this cell first

In [ ]:

# ----------------------------- INPUTS -----------------------------
# Raw integer count matrix used by MAGeCK (preferred).
COUNT_PATH = Path(
    "/oak/stanford/groups/engreitz/Users/chloemc/GenomeWideValidation/"
    "MageckAnalysis/mageck_sgRNA_readcount.txt"
)

# Fallback raw count matrix used by the earlier U-test workflow.
FALLBACK_COUNT_PATH = Path(
    "/oak/stanford/groups/engreitz/Users/chloemc/GenomeWideValidation/"
    "UTestAnalysis/guideCounts.csv"
)

# Set this to a CSV/TSV file after reviewing the automatically generated sample sheet.
# Required columns: sample, biological_replicate, fraction, technical_replicate, include
# Allowed fractions: POS, NEG, D0, OTHER
SAMPLE_SHEET_PATH: Optional[Path] = None

# Optional: text/CSV/TSV file containing one explicit control guide ID per row.
# Leave as None to detect safe-targeting and non-targeting guides from annotations.
EXPLICIT_CONTROL_GUIDES_PATH: Optional[Path] = None

# Columns that are numeric but are not sample count columns, if any.
EXCLUDE_COUNT_COLUMNS: list[str] = []

# ----------------------------- OUTPUT -----------------------------
USER = os.environ.get("USER", "YOUR_USERNAME")
OUTPUT_DIR = Path(
    f"/oak/stanford/groups/engreitz/Users/{USER}/GenomeWideValidation/"
    "guide_level_EC_efficiency_edgeR"
)

# ---------------------- CONTROL IDENTIFICATION --------------------
# Applied to guide ID + target/gene annotation. Edit if the controls use other names.
CONTROL_REGEX = re.compile(
    r"(?i)(non[-_ ]?target(?:ing)?|(?:^|[^a-z0-9])ntc(?:[^a-z0-9]|$)|"
    r"safe[-_ ]?target(?:ing)?|negative[-_ ]?control|neg[-_ ]?control|safe[-_ ]?harbor)"
)
MIN_CONTROL_GUIDES = 20
MIN_CONTROL_TOTAL_D3_COUNT = 20

# -------------------------- COUNT FILTERS -------------------------
# A guide can be strongly depleted in one fraction and still pass because filtering is
# based on total D3 representation, not a minimum count in both fractions.
MIN_TOTAL_D3_COUNT = 20
MIN_SAMPLES_WITH_COUNT = 2
MIN_COUNT_IN_SAMPLE = 5
PSEUDOCOUNT_FOR_REPLICATE_PLOTS = 0.5  # only for descriptive replicate LFCs

# -------------------------- SIGNIFICANCE --------------------------
FDR_THRESHOLD = 0.05
RUN_EDGER = True
INSTALL_EDGER_IF_MISSING = False

# -------------------- OPTIONAL ABSOLUTE CALIBRATION ---------------
# Keep None unless a defensible EC fraction among control-guide cells is available.
# Example: 0.80 means 80% control EC efficiency.
ASSUMED_CONTROL_EC_FRACTION: Optional[float] = None

# Sensitivity values are exported separately and are explicitly assumption-dependent.
SENSITIVITY_CONTROL_EC_FRACTIONS = [0.50, 0.60, 0.70, 0.80, 0.90]

# Optional labels for plots.
GENES_TO_LABEL = [
    "KDR", "ETV2", "PLCG1", "BMPR1A", "ACVR2A", "SMAD1", "SMAD4", "SMAD5",
    "NF2", "LATS2", "SAV1", "AMOTL2", "TAOK1", "PRKCD", "MIXL1", "EOMES"
]

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Output directory:", OUTPUT_DIR)


## 2. Helper functions

In [ ]:

def natural_sort_key(value: object):
    """Sort strings containing numbers in human order."""
    text = str(value)
    return [int(x) if x.isdigit() else x.lower() for x in re.split(r"(\d+)", text)]


def read_table_auto(path: Path) -> pd.DataFrame:
    """Read CSV/TSV/whitespace-delimited text with conservative delimiter detection."""
    if not path.exists():
        raise FileNotFoundError(path)

    suffixes = "".join(path.suffixes).lower()
    if suffixes.endswith(".csv") or suffixes.endswith(".csv.gz"):
        df = pd.read_csv(path)
    else:
        # MAGeCK inputs are normally tab-delimited.
        df = pd.read_csv(path, sep="\t", low_memory=False)
        if df.shape[1] == 1:
            df = pd.read_csv(path, sep=None, engine="python", low_memory=False)

    df.columns = [str(c).strip() for c in df.columns]
    return df


def find_column(columns: Iterable[str], candidates: Iterable[str]) -> Optional[str]:
    normalized = {re.sub(r"[^a-z0-9]", "", str(c).lower()): c for c in columns}
    for candidate in candidates:
        key = re.sub(r"[^a-z0-9]", "", candidate.lower())
        if key in normalized:
            return normalized[key]
    return None


def detect_count_table_structure(df: pd.DataFrame):
    """Detect guide ID, target annotation, and raw-count sample columns."""
    guide_col = find_column(
        df.columns,
        ["sgRNA", "sgrna", "guide", "guide_id", "guideid", "id", "Unnamed: 0"],
    )
    if guide_col is None:
        first = df.columns[0]
        if df[first].astype(str).is_unique:
            guide_col = first
        else:
            raise ValueError("Could not identify a unique guide-ID column.")

    gene_col = find_column(
        df.columns,
        ["Gene", "gene", "target", "target_gene", "targetgene", "element", "annotation"],
    )

    excluded = {guide_col, *EXCLUDE_COUNT_COLUMNS}
    if gene_col is not None:
        excluded.add(gene_col)

    count_columns: list[str] = []
    extra_metadata: list[str] = []
    converted: dict[str, pd.Series] = {}

    for col in df.columns:
        if col in excluded:
            continue
        numeric = pd.to_numeric(df[col], errors="coerce")
        # A count column should be almost entirely numeric.
        if numeric.notna().mean() >= 0.98:
            count_columns.append(col)
            converted[col] = numeric
        else:
            extra_metadata.append(col)

    if len(count_columns) < 4:
        raise ValueError(
            f"Detected only {len(count_columns)} numeric sample columns. "
            "Inspect the table and add non-count numeric fields to EXCLUDE_COUNT_COLUMNS."
        )

    counts = pd.DataFrame(converted, index=df.index)
    if counts.isna().any().any():
        bad = counts.columns[counts.isna().any()].tolist()
        raise ValueError(f"Missing/non-numeric values in count columns: {bad}")
    if (counts < 0).any().any():
        raise ValueError("Negative values were found; raw guide counts must be non-negative.")

    fractional = np.abs(counts.to_numpy(dtype=float) - np.rint(counts.to_numpy(dtype=float)))
    n_fractional = int((fractional > 1e-6).sum())
    if n_fractional > 0:
        raise ValueError(
            f"Found {n_fractional:,} non-integer entries. edgeR requires raw integer counts. "
            "Use mageck_sgRNA_readcount.txt or the unnormalized guideCounts.csv, not a "
            "MAGeCK summary or normalized-count table."
        )

    guide_ids = df[guide_col].astype(str).str.strip()
    if guide_ids.duplicated().any():
        duplicated = guide_ids[guide_ids.duplicated(keep=False)].head(20).tolist()
        raise ValueError(f"Guide IDs are not unique. Examples: {duplicated}")

    annotation = pd.DataFrame({"guide_id": guide_ids})
    if gene_col is not None:
        annotation["target"] = df[gene_col].astype(str).str.strip()
    else:
        annotation["target"] = ""
    for col in extra_metadata:
        annotation[col] = df[col]

    counts = counts.round().astype(np.int64)
    counts.index = guide_ids
    annotation.index = guide_ids

    return annotation, counts, guide_col, gene_col, extra_metadata


def infer_fraction(sample: str) -> str:
    text = str(sample).lower()
    # D0 before NEG/POS to avoid accidental matches in compound labels.
    if re.search(r"(?:^|[^a-z0-9])(?:d0|day[-_ ]?0)(?:[^a-z0-9]|$)", text):
        return "D0"
    pos_patterns = [
        r"cdh5[-_ ]?(?:pos|positive)", r"d3[-_ ]?(?:pos|positive)",
        r"(?:cdh5|d3|ec)[-_ ]?\+", r"(?:^|[_\-. ])d3p(?:$|[_\-. ])",
        r"macs[-_ ]?(?:pos|positive)", r"ec[-_ ]?(?:pos|positive)",
        r"(?:^|[_\-. ])(?:pos|positive)(?:$|[_\-. ])",
    ]
    neg_patterns = [
        r"cdh5[-_ ]?(?:neg|negative)", r"d3[-_ ]?(?:neg|negative)",
        r"(?:cdh5|d3|ec)[-_ ]?-", r"(?:^|[_\-. ])d3n(?:$|[_\-. ])",
        r"macs[-_ ]?(?:neg|negative)", r"ec[-_ ]?(?:neg|negative)",
        r"(?:^|[_\-. ])(?:neg|negative)(?:$|[_\-. ])",
    ]
    if any(re.search(p, text) for p in pos_patterns):
        return "POS"
    if any(re.search(p, text) for p in neg_patterns):
        return "NEG"
    return "OTHER"


def infer_biological_replicate(sample: str) -> str:
    """Use only strong biological-replicate tokens; generic PCR replicate numbers are unsafe."""
    text = str(sample)
    patterns = [
        r"(?i)(?:^|[_\-. ])(?:bio(?:logical)?[-_ ]?rep(?:licate)?|biorep|br)[-_ ]?0*(\d+)(?:$|[_\-. ])",
        r"(?i)(?:^|[_\-. ])P0*([12])(?:$|[_\-. ])",
        r"(?i)(?:^|[_\-. ])B0*([12])(?:$|[_\-. ])",
    ]
    for pattern in patterns:
        match = re.search(pattern, text)
        if match:
            return f"B{int(match.group(1))}"
    return ""


def infer_technical_replicate(sample: str) -> str:
    text = str(sample)
    patterns = [
        r"(?i)(?:pcr|tech|technical|lib|library|lane)[-_ ]?0*(\d+)",
        r"(?i)(?:^|[_\-. ])T0*(\d+)(?:$|[_\-. ])",
    ]
    for pattern in patterns:
        match = re.search(pattern, text)
        if match:
            return str(int(match.group(1)))
    return str(sample)


def parse_bool(value: object) -> bool:
    if isinstance(value, bool):
        return value
    return str(value).strip().lower() in {"true", "t", "1", "yes", "y"}


def load_or_make_sample_sheet(sample_columns: list[str], output_dir: Path) -> pd.DataFrame:
    template_path = output_dir / "sample_sheet_to_edit.csv"

    if SAMPLE_SHEET_PATH is not None:
        path = Path(SAMPLE_SHEET_PATH)
        sheet = read_table_auto(path)
        source = path
    else:
        sheet = pd.DataFrame({"sample": sample_columns})
        sheet["biological_replicate"] = sheet["sample"].map(infer_biological_replicate)
        sheet["fraction"] = sheet["sample"].map(infer_fraction)
        sheet["technical_replicate"] = sheet["sample"].map(infer_technical_replicate)
        sheet["include"] = sheet["fraction"].isin(["POS", "NEG", "D0"])
        sheet["notes"] = ""
        sheet.to_csv(template_path, index=False)
        source = template_path

    required = {"sample", "biological_replicate", "fraction", "technical_replicate", "include"}
    missing = required.difference(sheet.columns)
    if missing:
        raise ValueError(f"Sample sheet is missing required columns: {sorted(missing)}")

    sheet = sheet.copy()
    sheet["sample"] = sheet["sample"].astype(str).str.strip()
    sheet["biological_replicate"] = sheet["biological_replicate"].fillna("").astype(str).str.strip()
    sheet["fraction"] = sheet["fraction"].fillna("OTHER").astype(str).str.upper().str.strip()
    sheet["technical_replicate"] = sheet["technical_replicate"].fillna("").astype(str).str.strip()
    sheet["include"] = sheet["include"].map(parse_bool)

    unknown_samples = sorted(set(sheet["sample"]) - set(sample_columns))
    missing_samples = sorted(set(sample_columns) - set(sheet["sample"]))
    if unknown_samples:
        raise ValueError(f"Sample sheet names absent from count matrix: {unknown_samples[:20]}")
    if missing_samples:
        warnings.warn(f"Count columns absent from sample sheet will be excluded: {missing_samples[:20]}")
    if sheet["sample"].duplicated().any():
        raise ValueError("Each count-matrix sample must occur at most once in the sample sheet.")

    bad_fraction = sorted(set(sheet["fraction"]) - {"POS", "NEG", "D0", "OTHER"})
    if bad_fraction:
        raise ValueError(f"Unsupported fraction labels: {bad_fraction}")

    d3 = sheet.loc[sheet["include"] & sheet["fraction"].isin(["POS", "NEG"])].copy()
    if d3.empty:
        raise RuntimeError(
            f"No included POS/NEG samples were identified. Edit {source}, set SAMPLE_SHEET_PATH, "
            "and rerun this cell."
        )
    if (d3["biological_replicate"] == "").any():
        unresolved = d3.loc[d3["biological_replicate"] == "", "sample"].tolist()
        raise RuntimeError(
            "Biological replicate could not be safely inferred for these POS/NEG samples:\n"
            + "\n".join(f"  - {x}" for x in unresolved[:30])
            + f"\nEdit {source}, set SAMPLE_SHEET_PATH to that file, and rerun."
        )

    table = d3.groupby(["biological_replicate", "fraction"]).size().unstack(fill_value=0)
    for needed in ["NEG", "POS"]:
        if needed not in table.columns:
            table[needed] = 0
    unmatched = table[(table["NEG"] == 0) | (table["POS"] == 0)]
    if not unmatched.empty:
        raise RuntimeError(
            "Every included biological replicate must contain both NEG and POS libraries.\n"
            f"Current mapping:\n{table}\nEdit {source} and rerun."
        )
    if table.shape[0] < 2:
        raise RuntimeError(
            "At least two matched biological replicates are required for guide-level significance. "
            f"Current mapping has {table.shape[0]}."
        )

    print("Sample-sheet source:", source)
    print("Matched POS/NEG technical-library counts:")
    display(table.reindex(sorted(table.index, key=natural_sort_key)))
    return sheet


def load_explicit_control_ids(path: Path) -> set[str]:
    # Plain one-ID-per-line text files are accepted without requiring a header.
    if path.suffix.lower() in {".txt", ".list"}:
        values = []
        for line in path.read_text().splitlines():
            token = line.split("\t", 1)[0].split(",", 1)[0].strip()
            if token and not token.startswith("#"):
                values.append(token)
        if values and re.sub(r"[^a-z0-9]", "", values[0].lower()) in {
            "guideid", "guide", "sgrna", "id"
        }:
            values = values[1:]
        return set(values)

    data = read_table_auto(path)
    if data.shape[1] == 0:
        return set()
    candidate = find_column(data.columns, ["guide_id", "guide", "sgRNA", "sgrna", "id"])
    if candidate is None:
        candidate = data.columns[0]
    return set(data[candidate].dropna().astype(str).str.strip())


def classify_control(annotation: pd.DataFrame) -> pd.DataFrame:
    out = annotation.copy()
    combined = (out["guide_id"].fillna("") + " " + out["target"].fillna("")).astype(str)

    if EXPLICIT_CONTROL_GUIDES_PATH is not None:
        explicit = load_explicit_control_ids(Path(EXPLICIT_CONTROL_GUIDES_PATH))
        out["is_control"] = out["guide_id"].isin(explicit)
        source = f"explicit list: {EXPLICIT_CONTROL_GUIDES_PATH}"
    else:
        out["is_control"] = combined.str.contains(CONTROL_REGEX, regex=True, na=False)
        source = "CONTROL_REGEX"

    lower = combined.str.lower()
    out["control_class"] = np.select(
        [
            lower.str.contains(r"non[-_ ]?target|(?:^|[^a-z0-9])ntc(?:[^a-z0-9]|$)", regex=True),
            lower.str.contains(r"safe[-_ ]?target|safe[-_ ]?harbor", regex=True),
            out["is_control"],
        ],
        ["non_targeting", "safe_targeting", "other_control"],
        default="targeting",
    )

    n = int(out["is_control"].sum())
    print(f"Detected {n} control guides using {source}.")
    display(out.loc[out["is_control"], ["guide_id", "target", "control_class"]].head(20))
    if n < MIN_CONTROL_GUIDES:
        raise RuntimeError(
            f"Only {n} controls were detected; at least {MIN_CONTROL_GUIDES} are required. "
            "Edit CONTROL_REGEX or provide EXPLICIT_CONTROL_GUIDES_PATH."
        )
    return out


def bh_adjust(p_values: pd.Series) -> pd.Series:
    """Benjamini-Hochberg adjusted p-values, preserving the original index."""
    p = pd.to_numeric(p_values, errors="coerce")
    result = pd.Series(np.nan, index=p.index, dtype=float)
    valid = p.notna()
    values = p.loc[valid].to_numpy(dtype=float)
    if values.size == 0:
        return result
    order = np.argsort(values)
    ranked = values[order]
    adjusted = ranked * len(ranked) / np.arange(1, len(ranked) + 1)
    adjusted = np.minimum.accumulate(adjusted[::-1])[::-1]
    adjusted = np.clip(adjusted, 0, 1)
    restored = np.empty_like(adjusted)
    restored[order] = adjusted
    result.loc[valid] = restored
    return result


def clean_gene_symbol(target: object) -> str:
    text = str(target).strip()
    # Preserve original target in the output; this is only a display label.
    text = re.sub(r"_-?P\d+(?:P\d+)*$", "", text)
    text = re.sub(r"[-_]P\d+(?:P\d+)*$", "", text)
    return text


def save_figure(fig: plt.Figure, stem: Path):
    fig.savefig(stem.with_suffix(".pdf"), bbox_inches="tight")
    fig.savefig(stem.with_suffix(".png"), dpi=300, bbox_inches="tight")


## 3. Load and validate the raw guide-count matrix

In [ ]:

if COUNT_PATH.exists():
    count_path_used = COUNT_PATH
elif FALLBACK_COUNT_PATH.exists():
    count_path_used = FALLBACK_COUNT_PATH
    warnings.warn(f"Preferred MAGeCK count table not found; using fallback: {count_path_used}")
else:
    raise FileNotFoundError(
        "Neither raw count path exists:\n"
        f"  {COUNT_PATH}\n"
        f"  {FALLBACK_COUNT_PATH}\n"
        "Edit the configuration cell if the files were moved."
    )

raw_table = read_table_auto(count_path_used)
annotation, raw_counts, guide_id_col, gene_col, extra_metadata_cols = detect_count_table_structure(raw_table)

print("Input:", count_path_used)
print("Rows (guides):", f"{raw_counts.shape[0]:,}")
print("Numeric sample columns:", raw_counts.shape[1])
print("Guide-ID column:", guide_id_col)
print("Target column:", gene_col)
print("Other annotation columns:", extra_metadata_cols)
print("Total reads:", f"{int(raw_counts.to_numpy().sum()):,}")

display(annotation.head())
display(raw_counts.head())



## 4. Map count columns to biological samples

The notebook attempts only conservative inference. It does not treat generic numbered PCR libraries as biological replicates.

When inference fails:

1. open `sample_sheet_to_edit.csv` in the output directory;
2. assign each column to biological replicate `B1`, `B2`, etc.;
3. use fraction `POS`, `NEG`, `D0`, or `OTHER`;
4. mark unusable libraries with `include = FALSE`;
5. set `SAMPLE_SHEET_PATH` in the configuration cell to the edited file;
6. rerun from this section.


In [ ]:

sample_sheet = load_or_make_sample_sheet(raw_counts.columns.tolist(), OUTPUT_DIR)
display(sample_sheet)

sample_sheet.to_csv(OUTPUT_DIR / "sample_sheet_used.tsv", sep="\t", index=False)


## 5. Identify negative-control guides

In [ ]:

annotation = classify_control(annotation)
annotation["gene_symbol_display"] = annotation["target"].map(clean_gene_symbol)

control_summary = (
    annotation.groupby("control_class", dropna=False)
    .size()
    .rename("n_guides")
    .reset_index()
)
display(control_summary)
annotation.to_csv(OUTPUT_DIR / "guide_annotation_with_control_class.tsv", sep="\t", index=False)


## 6. Collapse technical PCR/sequencing replicates

In [ ]:

model_sheet = sample_sheet.loc[
    sample_sheet["include"] & sample_sheet["fraction"].isin(["NEG", "POS"])
].copy()

replicates = sorted(model_sheet["biological_replicate"].unique(), key=natural_sort_key)
collapsed = {}
collapsed_meta = []

for replicate in replicates:
    for fraction in ["NEG", "POS"]:
        members = model_sheet.loc[
            (model_sheet["biological_replicate"] == replicate)
            & (model_sheet["fraction"] == fraction),
            "sample",
        ].tolist()
        if not members:
            raise RuntimeError(f"Missing {fraction} libraries for {replicate}")
        output_name = f"{replicate}__{fraction}"
        collapsed[output_name] = raw_counts[members].sum(axis=1)
        collapsed_meta.append(
            {
                "sample": output_name,
                "biological_replicate": replicate,
                "fraction": fraction,
                "n_technical_libraries": len(members),
                "source_libraries": ";".join(members),
            }
        )

collapsed_counts = pd.DataFrame(collapsed, index=raw_counts.index).astype(np.int64)
collapsed_samples = pd.DataFrame(collapsed_meta)
model_columns = collapsed_samples["sample"].tolist()
collapsed_counts = collapsed_counts[model_columns]

# D0 is retained only as descriptive baseline coverage; it is not used in the POS-vs-NEG model.
d0_columns = sample_sheet.loc[
    sample_sheet["include"] & (sample_sheet["fraction"] == "D0"), "sample"
].tolist()
if d0_columns:
    d0_total = raw_counts[d0_columns].sum(axis=1).rename("D0_total_count")
else:
    d0_total = pd.Series(np.nan, index=raw_counts.index, name="D0_total_count")

collapsed_counts.to_csv(OUTPUT_DIR / "collapsed_D3_POS_NEG_raw_counts.tsv", sep="\t", index_label="guide_id")
collapsed_samples.to_csv(OUTPUT_DIR / "collapsed_D3_sample_metadata.tsv", sep="\t", index=False)

display(collapsed_samples)
display(collapsed_counts.head())


## 7. Count-level QC and controls used for normalization

In [ ]:

library_sizes = collapsed_counts.sum(axis=0).rename("raw_library_size").to_frame()
library_sizes["control_reads"] = collapsed_counts.loc[annotation["is_control"]].sum(axis=0)
library_sizes["control_read_fraction"] = (
    library_sizes["control_reads"] / library_sizes["raw_library_size"]
)
library_sizes = library_sizes.join(collapsed_samples.set_index("sample"))
library_sizes.to_csv(OUTPUT_DIR / "library_size_QC.tsv", sep="\t")
display(library_sizes)

count_qc = pd.DataFrame(index=collapsed_counts.index)
count_qc["D3_total_count"] = collapsed_counts.sum(axis=1)
count_qc["n_D3_samples_count_ge_threshold"] = (collapsed_counts >= MIN_COUNT_IN_SAMPLE).sum(axis=1)
count_qc["passes_count_QC"] = (
    (count_qc["D3_total_count"] >= MIN_TOTAL_D3_COUNT)
    & (count_qc["n_D3_samples_count_ge_threshold"] >= MIN_SAMPLES_WITH_COUNT)
)
count_qc["D0_total_count"] = d0_total

control_for_norm = (
    annotation["is_control"]
    & (count_qc["D3_total_count"] >= MIN_CONTROL_TOTAL_D3_COUNT)
)
control_ids_for_norm = annotation.index[control_for_norm].tolist()
print("Control guides retained for normalization:", len(control_ids_for_norm))
if len(control_ids_for_norm) < MIN_CONTROL_GUIDES:
    raise RuntimeError(
        f"Only {len(control_ids_for_norm)} control guides pass the normalization count threshold. "
        "Inspect control annotation and counts before lowering the threshold."
    )

(OUTPUT_DIR / "control_guides_for_normalization.txt").write_text(
    "\n".join(control_ids_for_norm) + "\n"
)
count_qc.to_csv(OUTPUT_DIR / "guide_count_QC.tsv", sep="\t", index_label="guide_id")

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(np.arange(library_sizes.shape[0]), library_sizes["raw_library_size"].to_numpy())
ax.set_xticks(np.arange(library_sizes.shape[0]))
ax.set_xticklabels(library_sizes.index, rotation=45, ha="right")
ax.set_ylabel("Raw guide reads")
ax.set_title("Collapsed D3 library sizes")
fig.tight_layout()
save_figure(fig, OUTPUT_DIR / "QC_library_sizes")
plt.show()

# Correlation on log2 counts-per-million for a broad library-level check.
cpm = collapsed_counts.divide(collapsed_counts.sum(axis=0), axis=1) * 1e6
log_cpm = np.log2(cpm + 0.5)
corr = log_cpm.corr(method="pearson")
corr.to_csv(OUTPUT_DIR / "QC_sample_logCPM_correlations.tsv", sep="\t")

fig, ax = plt.subplots(figsize=(6, 5.5))
im = ax.imshow(corr.to_numpy(), aspect="auto")
ax.set_xticks(np.arange(corr.shape[1]))
ax.set_yticks(np.arange(corr.shape[0]))
ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticklabels(corr.index)
ax.set_title("Sample correlation: log2 CPM")
fig.colorbar(im, ax=ax, label="Pearson r")
fig.tight_layout()
save_figure(fig, OUTPUT_DIR / "QC_sample_correlations")
plt.show()



## 8. edgeR quasi-likelihood model

For guide \(j\) and collapsed D3 sample \(s\), the fitted model is

\[
Y_{js}\sim \mathrm{NB}(\mu_{js},\phi_j),
\qquad
\log \mu_{js}
= \log N_s + \alpha_j + b_{j,\mathrm{replicate}(s)}
+ \beta_j I(s=\mathrm{POS}).
\]

`N_s` is an effective library size estimated from the safe-targeting/non-targeting control guides. `β_j / log(2)` is the `POS − NEG` log2 relative-odds effect.

The model uses raw integer counts. The earlier normalized decimal values shown in the generated R plotting files are not valid edgeR inputs.


In [ ]:

# Check R and edgeR. This cell does not silently install packages unless explicitly enabled.
rscript = shutil.which("Rscript")
if rscript is None:
    raise RuntimeError(
        "Rscript is not on PATH. On Sherlock, run `module spider R`, load an available R module, "
        "then restart or rerun this notebook."
    )

check = subprocess.run(
    [rscript, "-e", "cat(requireNamespace('edgeR', quietly=TRUE))"],
    capture_output=True,
    text=True,
)
edgeR_available = check.stdout.strip().upper() == "TRUE"
print("Rscript:", rscript)
print("edgeR available:", edgeR_available)

if not edgeR_available and INSTALL_EDGER_IF_MISSING:
    install_code = """
    if (!requireNamespace('BiocManager', quietly=TRUE))
        install.packages('BiocManager', repos='https://cloud.r-project.org')
    BiocManager::install('edgeR', ask=FALSE, update=FALSE)
    """
    subprocess.run([rscript, "-e", install_code], check=True)
    edgeR_available = True

if not edgeR_available:
    raise RuntimeError(
        "edgeR is not installed in the active R library. Install once with:\n\n"
        "Rscript -e \"if (!requireNamespace('BiocManager', quietly=TRUE)) "
        "install.packages('BiocManager', repos='https://cloud.r-project.org'); "
        "BiocManager::install('edgeR', ask=FALSE, update=FALSE)\"\n\n"
        "Then rerun this cell."
    )


In [ ]:

# Write model inputs.
edger_input_counts = OUTPUT_DIR / "edgeR_input_counts.tsv"
edger_input_samples = OUTPUT_DIR / "edgeR_input_samples.tsv"
edger_controls = OUTPUT_DIR / "control_guides_for_normalization.txt"
edger_results_path = OUTPUT_DIR / "edgeR_guide_results.tsv"
edger_norm_path = OUTPUT_DIR / "edgeR_normalization_factors.tsv"
edger_design_path = OUTPUT_DIR / "edgeR_design_matrix.tsv"
edger_session_path = OUTPUT_DIR / "edgeR_sessionInfo.txt"
edger_script_path = OUTPUT_DIR / "run_guide_level_edgeR.R"

collapsed_counts.to_csv(edger_input_counts, sep="\t", index_label="guide_id")
collapsed_samples[["sample", "biological_replicate", "fraction"]].to_csv(
    edger_input_samples, sep="\t", index=False
)

r_code = r'''
suppressPackageStartupMessages(library(edgeR))

args <- commandArgs(trailingOnly=TRUE)
counts_path <- args[[1]]
samples_path <- args[[2]]
controls_path <- args[[3]]
results_path <- args[[4]]
norm_path <- args[[5]]
design_path <- args[[6]]
session_path <- args[[7]]
min_total <- as.numeric(args[[8]])
min_samples <- as.numeric(args[[9]])
min_count <- as.numeric(args[[10]])

counts_df <- read.delim(counts_path, check.names=FALSE, stringsAsFactors=FALSE)
if (!"guide_id" %in% colnames(counts_df)) stop("Missing guide_id column")
rownames(counts_df) <- counts_df$guide_id
counts_df$guide_id <- NULL
counts <- as.matrix(counts_df)
storage.mode(counts) <- "integer"

samples <- read.delim(samples_path, check.names=FALSE, stringsAsFactors=FALSE)
if (!identical(colnames(counts), samples$sample)) {
    stop("Count columns and sample metadata are not in identical order")
}

control_ids <- scan(controls_path, what=character(), quiet=TRUE)
is_control <- rownames(counts) %in% control_ids
if (sum(is_control) < 20) stop("Too few control guides for normalization")

# TMM is computed on neutral controls only. The resulting effective control-library
# sizes are rescaled to the geometric mean of the full raw library sizes. The common
# scale is arbitrary and is absorbed by the guide intercepts; relative sizes matter.
control_y <- DGEList(counts=counts[is_control, , drop=FALSE])
control_y <- calcNormFactors(control_y, method="TMM")
control_effective <- with(control_y$samples, lib.size * norm.factors)
full_raw_library <- colSums(counts)
scale_target <- exp(mean(log(pmax(full_raw_library, 1))))
effective_library <- control_effective / exp(mean(log(control_effective))) * scale_target

# Build the DGEList with control-derived effective library sizes.
y <- DGEList(counts=counts)
y$samples$lib.size <- effective_library
y$samples$norm.factors <- rep(1, ncol(y))

samples$biological_replicate <- factor(samples$biological_replicate)
samples$fraction <- factor(samples$fraction, levels=c("NEG", "POS"))
design <- model.matrix(~ biological_replicate + fraction, data=samples)
rownames(design) <- samples$sample
if (qr(design)$rank < ncol(design)) stop("Design matrix is not full rank")
if (nrow(design) - ncol(design) < 1) {
    stop("No residual degrees of freedom; at least two matched biological replicates are required")
}

keep <- rowSums(counts) >= min_total & rowSums(counts >= min_count) >= min_samples
if (sum(keep) < 100) warning("Fewer than 100 guides passed count filtering")

y_fit <- y[keep, , keep.lib.sizes=TRUE]
y_fit <- estimateDisp(y_fit, design, robust=TRUE)
fit <- glmQLFit(y_fit, design, robust=TRUE)
coef_index <- which(colnames(design) == "fractionPOS")
if (length(coef_index) != 1) stop("Could not identify fractionPOS coefficient")
qlf <- glmQLFTest(fit, coef=coef_index)
tab <- topTags(qlf, n=Inf, sort.by="none")$table
tab$guide_id <- rownames(tab)

all_guides <- data.frame(
    guide_id=rownames(counts),
    passes_edgeR_filter=keep,
    stringsAsFactors=FALSE
)
all_guides <- merge(all_guides, tab, by="guide_id", all.x=TRUE, sort=FALSE)
all_guides <- all_guides[match(rownames(counts), all_guides$guide_id), ]
write.table(all_guides, results_path, sep="\t", quote=FALSE, row.names=FALSE)

norm_table <- data.frame(
    sample=colnames(counts),
    biological_replicate=samples$biological_replicate,
    fraction=samples$fraction,
    raw_library_size=full_raw_library,
    control_raw_library_size=colSums(counts[is_control, , drop=FALSE]),
    control_TMM_norm_factor=control_y$samples$norm.factors,
    effective_library_size=effective_library,
    stringsAsFactors=FALSE
)
write.table(norm_table, norm_path, sep="\t", quote=FALSE, row.names=FALSE)
write.table(
    data.frame(sample=rownames(design), design, check.names=FALSE),
    design_path, sep="\t", quote=FALSE, row.names=FALSE
)

sink(session_path)
print(sessionInfo())
sink()
'''

edger_script_path.write_text(r_code)

command = [
    rscript,
    str(edger_script_path),
    str(edger_input_counts),
    str(edger_input_samples),
    str(edger_controls),
    str(edger_results_path),
    str(edger_norm_path),
    str(edger_design_path),
    str(edger_session_path),
    str(MIN_TOTAL_D3_COUNT),
    str(MIN_SAMPLES_WITH_COUNT),
    str(MIN_COUNT_IN_SAMPLE),
]

print("Running:", " ".join(command[:2]), "[paths omitted]")
if RUN_EDGER:
    completed = subprocess.run(command, capture_output=True, text=True)
    print(completed.stdout)
    if completed.returncode != 0:
        print(completed.stderr)
        raise RuntimeError(f"edgeR failed with exit code {completed.returncode}")
else:
    print("RUN_EDGER=False; inputs and R script were written, but the model was not run.")


## 9. Assemble guide-level effects, significance, and replicate concordance

In [ ]:

if not edger_results_path.exists():
    raise FileNotFoundError(edger_results_path)

edger = pd.read_csv(edger_results_path, sep="\t")
norm = pd.read_csv(edger_norm_path, sep="\t")

results = annotation.reset_index(drop=True).merge(edger, on="guide_id", how="left", validate="one_to_one")
results = results.merge(count_qc.reset_index(names="guide_id"), on="guide_id", how="left", validate="one_to_one")

# Standardize edgeR column names while retaining the original fields.
results = results.rename(
    columns={
        "logFC": "log2_relative_CDH5_pos_odds",
        "logCPM": "edgeR_logCPM",
        "PValue": "p_value_edgeR_QL",
        "FDR": "edgeR_FDR_all_tested_guides",
        "F": "edgeR_QL_F",
    }
)

# Descriptive per-biological-replicate log2 ratios using the same effective-library offsets.
effective = norm.set_index("sample")["effective_library_size"].to_dict()
replicate_lfc_columns = []
for replicate in replicates:
    neg_col = f"{replicate}__NEG"
    pos_col = f"{replicate}__POS"
    if neg_col not in collapsed_counts or pos_col not in collapsed_counts:
        raise RuntimeError(f"Missing collapsed pair for {replicate}")
    lfc = (
        np.log2((collapsed_counts[pos_col] + PSEUDOCOUNT_FOR_REPLICATE_PLOTS) / effective[pos_col])
        - np.log2((collapsed_counts[neg_col] + PSEUDOCOUNT_FOR_REPLICATE_PLOTS) / effective[neg_col])
    )
    col = f"{replicate}_descriptive_log2_odds"
    results[col] = results["guide_id"].map(lfc)
    replicate_lfc_columns.append(col)

results["replicate_mean_descriptive_log2_odds"] = results[replicate_lfc_columns].mean(axis=1)
results["replicate_SD_descriptive_log2_odds"] = results[replicate_lfc_columns].std(axis=1, ddof=1)
results["replicate_min_log2_odds"] = results[replicate_lfc_columns].min(axis=1)
results["replicate_max_log2_odds"] = results[replicate_lfc_columns].max(axis=1)
results["replicate_direction_concordant"] = (
    (results[replicate_lfc_columns].gt(0).all(axis=1))
    | (results[replicate_lfc_columns].lt(0).all(axis=1))
)

results["relative_CDH5_pos_odds_pct_of_control"] = (
    100.0 * np.power(2.0, results["log2_relative_CDH5_pos_odds"])
)
results["effect_direction"] = np.select(
    [
        results["log2_relative_CDH5_pos_odds"] < 0,
        results["log2_relative_CDH5_pos_odds"] > 0,
    ],
    ["reduced_CDH5_pos_odds", "increased_CDH5_pos_odds"],
    default="no_change",
)

# BH correction across QC-passing TARGETING guides only.
testing_mask = (
    (~results["is_control"])
    & results["passes_edgeR_filter"].fillna(False)
    & results["p_value_edgeR_QL"].notna()
)
results["tested_targeting_guide"] = testing_mask
results["FDR_BH_targeting_guides"] = np.nan
results.loc[testing_mask, "FDR_BH_targeting_guides"] = bh_adjust(
    results.loc[testing_mask, "p_value_edgeR_QL"]
)
results["significant_FDR05"] = (
    testing_mask & (results["FDR_BH_targeting_guides"] < FDR_THRESHOLD)
)
results["high_confidence_guide"] = (
    results["significant_FDR05"]
    & results["replicate_direction_concordant"]
    & results["passes_count_QC"]
)

# Optional calibrated absolute percentage. The column name makes the assumption explicit.
if ASSUMED_CONTROL_EC_FRACTION is not None:
    p0 = float(ASSUMED_CONTROL_EC_FRACTION)
    if not (0 < p0 < 1):
        raise ValueError("ASSUMED_CONTROL_EC_FRACTION must lie strictly between 0 and 1.")
    odds_ratio = results["relative_CDH5_pos_odds_pct_of_control"] / 100.0
    results[f"estimated_EC_pct_assuming_control_{100*p0:g}pct"] = (
        100.0 * (odds_ratio * p0) / ((1.0 - p0) + odds_ratio * p0)
    )

# Add collapsed counts for auditability.
for col in collapsed_counts.columns:
    results[f"count_{col}"] = results["guide_id"].map(collapsed_counts[col])

# Put the most important columns first.
front = [
    "guide_id", "target", "gene_symbol_display", "is_control", "control_class",
    "log2_relative_CDH5_pos_odds", "relative_CDH5_pos_odds_pct_of_control",
    "p_value_edgeR_QL", "FDR_BH_targeting_guides", "tested_targeting_guide", "significant_FDR05",
    "replicate_direction_concordant", "high_confidence_guide", "effect_direction",
    "passes_count_QC", "D3_total_count", "D0_total_count",
] + replicate_lfc_columns
front = [c for c in front if c in results.columns]
remaining = [c for c in results.columns if c not in front]
results = results[front + remaining]

results = results.sort_values(
    ["is_control", "FDR_BH_targeting_guides", "log2_relative_CDH5_pos_odds"],
    ascending=[True, True, True],
    na_position="last",
).reset_index(drop=True)

display(results.head(30))


## 10. Assumption-dependent absolute EC% sensitivity table

In [ ]:

# This table is deliberately separate from the primary results.
base = results.loc[
    ~results["is_control"],
    ["guide_id", "target", "log2_relative_CDH5_pos_odds", "relative_CDH5_pos_odds_pct_of_control"],
].copy()

sensitivity_parts = []
for p0 in SENSITIVITY_CONTROL_EC_FRACTIONS:
    odds_ratio = base["relative_CDH5_pos_odds_pct_of_control"] / 100.0
    part = base.copy()
    part["assumed_control_EC_fraction"] = p0
    part["estimated_guide_EC_pct_under_assumption"] = (
        100.0 * (odds_ratio * p0) / ((1.0 - p0) + odds_ratio * p0)
    )
    sensitivity_parts.append(part)

absolute_sensitivity = pd.concat(sensitivity_parts, ignore_index=True)
absolute_sensitivity.to_csv(
    OUTPUT_DIR / "guide_level_absolute_EC_pct_sensitivity_ASSUMPTION_DEPENDENT.tsv",
    sep="\t", index=False,
)
display(absolute_sensitivity.head(15))


## 11. Result summary and QC diagnostics

In [ ]:

targeting = results.loc[~results["is_control"]].copy()
summary = pd.Series(
    {
        "n_guides_total": len(results),
        "n_targeting_guides": len(targeting),
        "n_control_guides": int(results["is_control"].sum()),
        "n_targeting_guides_tested": int(targeting["tested_targeting_guide"].sum()),
        "n_significant_FDR05": int(targeting["significant_FDR05"].sum()),
        "n_high_confidence": int(targeting["high_confidence_guide"].sum()),
        "n_significant_reduced_odds": int(
            (targeting["significant_FDR05"] & (targeting["log2_relative_CDH5_pos_odds"] < 0)).sum()
        ),
        "n_significant_increased_odds": int(
            (targeting["significant_FDR05"] & (targeting["log2_relative_CDH5_pos_odds"] > 0)).sum()
        ),
        "fraction_tested_with_replicate_direction_concordance": float(
            targeting.loc[targeting["tested_targeting_guide"], "replicate_direction_concordant"].mean()
        ) if targeting["tested_targeting_guide"].any() else np.nan,
    },
    name="value",
)
display(summary.to_frame())
summary.to_csv(OUTPUT_DIR / "analysis_summary.tsv", sep="\t", header=True)

# Compare safe-targeting and non-targeting control distributions when both are present.
control_effect_summary = (
    results.loc[results["is_control"]]
    .groupby("control_class")["log2_relative_CDH5_pos_odds"]
    .agg(["count", "median", "mean", "std"])
)
display(control_effect_summary)
control_effect_summary.to_csv(OUTPUT_DIR / "control_effect_QC.tsv", sep="\t")

if {"safe_targeting", "non_targeting"}.issubset(control_effect_summary.index):
    delta = abs(
        control_effect_summary.loc["safe_targeting", "median"]
        - control_effect_summary.loc["non_targeting", "median"]
    )
    print("Absolute median safe-vs-NTC log2-odds difference:", round(float(delta), 4))
    if delta > 0.5:
        warnings.warn(
            "Safe-targeting and non-targeting controls differ by >0.5 log2 units. "
            "Inspect control classes before interpreting pooled-control normalization."
        )


## 12. Publication-oriented diagnostic plots

In [ ]:

plot_df = targeting.loc[targeting["passes_edgeR_filter"].fillna(False)].copy()
plot_df["minus_log10_FDR"] = -np.log10(
    plot_df["FDR_BH_targeting_guides"].clip(lower=np.nextafter(0, 1))
)

# Volcano plot.
fig, ax = plt.subplots(figsize=(7.2, 6.0))
non_sig = plot_df.loc[~plot_df["significant_FDR05"]]
sig = plot_df.loc[plot_df["significant_FDR05"]]
ax.scatter(
    non_sig["log2_relative_CDH5_pos_odds"], non_sig["minus_log10_FDR"],
    s=10, alpha=0.35, label="FDR ≥ 0.05"
)
ax.scatter(
    sig["log2_relative_CDH5_pos_odds"], sig["minus_log10_FDR"],
    s=18, alpha=0.8, label="FDR < 0.05"
)
ax.axvline(0, linewidth=0.8, linestyle="--")
ax.axhline(-np.log10(FDR_THRESHOLD), linewidth=0.8, linestyle="--")
ax.set_xlabel("Guide-level log2 relative CDH5+ odds (POS − NEG)")
ax.set_ylabel("−log10(BH FDR)")
ax.set_title("Validation screen: guide-level EC differentiation effects")
ax.legend(frameon=False)

label_candidates = (
    sig.loc[sig["gene_symbol_display"].isin(GENES_TO_LABEL)]
    .sort_values("FDR_BH_targeting_guides")
    .drop_duplicates("gene_symbol_display")
    .copy()
)
for _, row in label_candidates.iterrows():
    ax.annotate(
        f"{row['gene_symbol_display']}\n{row['guide_id']}",
        (row["log2_relative_CDH5_pos_odds"], row["minus_log10_FDR"]),
        xytext=(3, 3), textcoords="offset points", fontsize=6,
    )
fig.tight_layout()
save_figure(fig, OUTPUT_DIR / "guide_level_volcano")
plt.show()

# Biological-replicate concordance. For >2 replicates, show first two and export all columns.
if len(replicate_lfc_columns) >= 2:
    x_col, y_col = replicate_lfc_columns[:2]
    fig, ax = plt.subplots(figsize=(6.2, 6.0))
    ax.scatter(
        targeting[x_col], targeting[y_col],
        s=10, alpha=0.45,
    )
    limits = np.nanpercentile(
        np.concatenate([targeting[x_col].to_numpy(), targeting[y_col].to_numpy()]),
        [0.5, 99.5],
    )
    ax.plot(limits, limits, linestyle="--", linewidth=0.8)
    ax.axhline(0, linewidth=0.6)
    ax.axvline(0, linewidth=0.6)
    ax.set_xlim(limits)
    ax.set_ylim(limits)
    ax.set_xlabel(x_col)
    ax.set_ylabel(y_col)
    valid = targeting[[x_col, y_col]].dropna()
    r = valid.corr().iloc[0, 1] if len(valid) > 2 else np.nan
    ax.set_title(f"Guide-effect concordance across biological replicates\nPearson r = {r:.3f}")
    fig.tight_layout()
    save_figure(fig, OUTPUT_DIR / "guide_effect_replicate_concordance")
    plt.show()

# Control versus targeting effect distributions.
fig, ax = plt.subplots(figsize=(7.2, 4.8))
ax.hist(
    results.loc[results["is_control"], "log2_relative_CDH5_pos_odds"].dropna(),
    bins=40, alpha=0.55, density=True, label="controls"
)
ax.hist(
    targeting["log2_relative_CDH5_pos_odds"].dropna(),
    bins=60, alpha=0.45, density=True, label="targeting"
)
ax.axvline(0, linewidth=0.8, linestyle="--")
ax.set_xlabel("Guide-level log2 relative CDH5+ odds")
ax.set_ylabel("Density")
ax.set_title("Control-centering diagnostic")
ax.legend(frameon=False)
fig.tight_layout()
save_figure(fig, OUTPUT_DIR / "control_vs_targeting_effect_distribution")
plt.show()


## 13. Optional guide-consistency summary by target

In [ ]:

# This is a descriptive summary only; it does not replace the guide-level statistical table.
gene_summary = (
    targeting.groupby(["target", "gene_symbol_display"], dropna=False)
    .agg(
        n_guides_total=("guide_id", "size"),
        n_guides_tested=("tested_targeting_guide", "sum"),
        median_log2_relative_CDH5_pos_odds=("log2_relative_CDH5_pos_odds", "median"),
        mean_relative_CDH5_pos_odds_pct=("relative_CDH5_pos_odds_pct_of_control", "mean"),
        n_guides_FDR05=("significant_FDR05", "sum"),
        n_high_confidence_guides=("high_confidence_guide", "sum"),
        fraction_replicate_direction_concordant=("replicate_direction_concordant", "mean"),
    )
    .reset_index()
)
gene_summary["fraction_guides_FDR05"] = (
    gene_summary["n_guides_FDR05"] / gene_summary["n_guides_tested"].replace(0, np.nan)
)
gene_summary = gene_summary.sort_values(
    ["n_high_confidence_guides", "n_guides_FDR05", "median_log2_relative_CDH5_pos_odds"],
    ascending=[False, False, True],
)
gene_summary.to_csv(OUTPUT_DIR / "target_level_guide_consistency_summary.tsv", sep="\t", index=False)
display(gene_summary.head(30))


## 14. Export final tables and an audit manifest

In [ ]:

all_results_path = OUTPUT_DIR / "guide_level_relative_CDH5_odds_with_significance.tsv"
significant_path = OUTPUT_DIR / "guide_level_significant_FDR05.tsv"
high_confidence_path = OUTPUT_DIR / "guide_level_high_confidence_FDR05_concordant.tsv"

results.to_csv(all_results_path, sep="\t", index=False)
results.loc[results["significant_FDR05"]].to_csv(significant_path, sep="\t", index=False)
results.loc[results["high_confidence_guide"]].to_csv(high_confidence_path, sep="\t", index=False)

manifest = {
    "count_path_used": str(count_path_used),
    "sample_sheet_path": str(SAMPLE_SHEET_PATH) if SAMPLE_SHEET_PATH is not None else str(OUTPUT_DIR / "sample_sheet_to_edit.csv"),
    "output_dir": str(OUTPUT_DIR),
    "n_guides": int(len(results)),
    "n_control_guides": int(results["is_control"].sum()),
    "n_targeting_guides_tested": int(targeting["tested_targeting_guide"].sum()),
    "n_significant_FDR05": int(results["significant_FDR05"].sum()),
    "n_high_confidence_guides": int(results["high_confidence_guide"].sum()),
    "fdr_threshold": FDR_THRESHOLD,
    "min_total_D3_count": MIN_TOTAL_D3_COUNT,
    "min_samples_with_count": MIN_SAMPLES_WITH_COUNT,
    "min_count_in_sample": MIN_COUNT_IN_SAMPLE,
    "assumed_control_EC_fraction": ASSUMED_CONTROL_EC_FRACTION,
    "primary_effect_definition": "100 * 2^(edgeR POS-vs-NEG log2FC), control-normalized",
    "absolute_EC_percentage_identifiable_without_calibration": False,
}
(OUTPUT_DIR / "analysis_manifest.json").write_text(json.dumps(manifest, indent=2) + "\n")

print("Primary results:", all_results_path)
print("Significant guides:", significant_path)
print("High-confidence guides:", high_confidence_path)
print("Sensitivity table:", OUTPUT_DIR / "guide_level_absolute_EC_pct_sensitivity_ASSUMPTION_DEPENDENT.tsv")
print("Audit manifest:", OUTPUT_DIR / "analysis_manifest.json")



## Reporting language

Recommended:

> Guide-level EC differentiation effects were quantified as the log2 ratio of guide abundance in D3 CDH5-positive versus CDH5-negative fractions, normalized to safe-targeting and non-targeting control guides. PCR/sequencing technical replicates were summed within each biological replicate and fraction. Guide counts were analyzed using an edgeR quasi-likelihood negative-binomial model including biological replicate and CDH5 fraction. Two-sided P values for the CDH5-positive versus CDH5-negative coefficient were adjusted across QC-passing targeting guides using the Benjamini–Hochberg method. Relative CDH5-positive differentiation odds were reported as \(100\times2^{\mathrm{log_2FC}}\), where 100% indicates the same odds as controls.

Do not write “guide-level EC percentage” unless an independently justified control EC fraction is supplied. The assumption-dependent sensitivity table is for interpretation only, not the primary result.
